In [2]:
# %pip install Pillow exifread opencv-python

## ver 1 pipeine

In [3]:
# class PhotoCleaner:
#     def __init__(self, blur_threshold=100.0, brightness_threshold=20.0):
#         self.blur_threshold = blur_threshold
#         self.brightness_threshold = brightness_threshold

#     def get_metadata(self, img_path):
#         """Extracts the timestamp from EXIF data."""
#         try:
#             image = Image.open(img_path)
#             exifdata = image._getexif()
#             if not exifdata:
#                 return None
            
#             for tag_id in exifdata:
#                 tag = TAGS.get(tag_id, tag_id)
#                 data = exifdata.get(tag_id)
#                 if tag == 'DateTimeOriginal':
#                     return data
#         except Exception as e:
#             print(f"Error reading metadata for {img_path}: {e}")
#         return None

#     def is_blurry(self, img_path):
#         """Calculates Laplacian variance to detect blur."""
#         image = cv2.imread(img_path)
#         gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         variance = cv2.Laplacian(gray, cv2.CV_64F).var()
#         return variance < self.blur_threshold, variance

#     def is_too_dark(self, img_path):
#         """Checks if the image is a pocket shot (too dark)."""
#         image = cv2.imread(img_path)
#         gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#         avg_brightness = np.mean(gray)
#         return avg_brightness < self.brightness_threshold, avg_brightness

#     def process_folder(self, folder_path):
#         clean_data = []
#         for filename in os.listdir(folder_path):
#             if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
#                 path = os.path.join(folder_path, filename)
                
#                 # Run Phase 1 checks
#                 blurry, v_score = self.is_blurry(path)
#                 dark, b_score = self.is_too_dark(path)
#                 timestamp = self.get_metadata(path)

#                 if not blurry and not dark:
#                     clean_data.append({
#                         "filename": filename,
#                         "timestamp": timestamp,
#                         "status": "Ready for DL"
#                     })
#                     print(f"✅ {filename}: Kept (Var: {v_score:.2f})")
#                 else:
#                     reason = "Blurry" if blurry else "Too Dark"
#                     print(f"❌ {filename}: Dropped ({reason})")
        
#         return clean_data

# # Usage
# # cleaner = PhotoCleaner()
# # processed_list = cleaner.process_folder('your_trip_photos_folder')

## optimized ver pipeline

In [4]:
# import os
# import cv2
# import numpy as np
# from concurrent.futures import ThreadPoolExecutor

# class OptimizedCleaner:
#     def __init__(self, blur_threshold=100.0, brightness_threshold=20.0):
#         self.blur_threshold = blur_threshold
#         self.brightness_threshold = brightness_threshold

#     def analyze_single_photo(self, path):
#         """Single-pass analysis of one photo."""
#         try:
#             # 1. Load image once
#             image = cv2.imread(path)
#             if image is None: return None
            
#             # 2. Convert to gray and Resize for speed
#             # Lower resolution is enough for blur/brightness detection
#             gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#             small_gray = cv2.resize(gray, (500, 500)) 

#             # 3. Math checks
#             v_score = cv2.Laplacian(small_gray, cv2.CV_64F).var()
#             b_score = np.mean(small_gray)

#             is_bad = v_score < self.blur_threshold or b_score < self.brightness_threshold
            
#             return {
#                 "path": path,
#                 "is_bad": is_bad,
#                 "v_score": v_score,
#                 "b_score": b_score
#             }
#         except Exception as e:
#             return {"path": path, "error": str(e)}

#     def parallel_process(self, folder_path):
#         """Uses ThreadPool to process images in parallel."""
#         files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
#                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
#         # Maximize CPU usage
#         with ThreadPoolExecutor() as executor:
#             results = list(executor.map(self.analyze_single_photo, files))
            
#         return [r for r in results if r and not r.get("is_bad")]

# # Usage
# # cleaner = OptimizedCleaner()
# # results = cleaner.parallel_process('my_trip_folder')

## hybrid ver pipeline

In [5]:
import os
import cv2
import exifread
import numpy as np
from concurrent.futures import ThreadPoolExecutor

class TripPreprocessor:
    def __init__(self, blur_threshold=100.0, brightness_threshold=20.0):
        self.blur_threshold = blur_threshold
        self.brightness_threshold = brightness_threshold

    def get_exif_data(self, path):
        """Reads EXIF without loading the full image pixel data."""
        with open(path, 'rb') as f:
            tags = exifread.process_file(f, stop_tag='DateTimeOriginal', details=False)
            timestamp = tags.get('EXIF DateTimeOriginal')
            return str(timestamp) if timestamp else None

    def analyze_photo(self, path):
        try:
            # 1. Quick Metadata Check
            timestamp = self.get_exif_data(path)
            
            # 2. Visual Quality Check (requires loading image)
            image = cv2.imread(path)
            if image is None: return None
            
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            small_gray = cv2.resize(gray, (500, 500)) 
            
            v_score = cv2.Laplacian(small_gray, cv2.CV_64F).var()
            b_score = np.mean(small_gray)

            # Drop if blurry or too dark
            if v_score < self.blur_threshold or b_score < self.brightness_threshold:
                return None

            return {
                "path": path,
                "timestamp": timestamp,
                "v_score": v_score,
                "b_score": b_score
            }
        except Exception as e:
            print(f"Skipping {path}: {e}")
            return None

    def run(self, folder_path):
        files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
                 if f.lower().endswith(('.jpg', '.jpeg'))]
        
        with ThreadPoolExecutor() as executor:
            results = list(executor.map(self.analyze_photo, files))
            
        # Filter out Nones and return clean list
        return [r for r in results if r is not None]

# Usage
# processor = TripPreprocessor()
# clean_data = processor.run('path_to_trip_photos')

## CLIP-Based Aesthetic Scorer

In [6]:
# pip install -U keras-hub keras

### ver 1 for single pic

In [7]:
# import keras
# import keras_hub
# import numpy as np
# from PIL import Image

# class AestheticScorer:
#     def __init__(self):
#         # Load pre-trained CLIP using KerasHub
#         self.model = keras_hub.models.CLIPBackbone.from_preset("clip_vit_base_patch32")
#         self.tokenizer = keras_hub.models.CLIPTokenizer.from_preset("clip_vit_base_patch32")
#         self.preprocessor = keras_hub.layers.CLIPImageConverter.from_preset("clip_vit_base_patch32")

#     def get_score(self, image_path):
#         # 1. Prepare Image
#         img = Image.open(image_path).convert("RGB")
#         img_input = self.preprocessor(np.array(img))
        
#         # 2. Prepare Prompts
#         prompts = ["a beautiful, high quality professional photo", 
#                    "a blurry, low quality amateur photo"]
#         token_ids = self.tokenizer(prompts)
        
#         # 3. Get Embeddings
#         outputs = self.model({
#             "token_ids": token_ids,
#             "images": np.expand_dims(img_input, axis=0)
#         })
        
#         # Calculate Cosine Similarity
#         # (Simplified: High similarity to prompt[0] = higher score)
#         logits_per_image = outputs["logits_per_image"]
#         probs = keras.activations.softmax(logits_per_image, axis=-1)
        
#         return float(probs[0][0]) # Probability that it matches the 'beautiful' prompt

# # Example Usage
# # scorer = AestheticScorer()
# # score = scorer.get_score("trip_photo_1.jpg")
# # print(f"Aesthetic Score: {score:.2f}")

### ver2 with batch processing

In [8]:
import keras
import keras_hub
import numpy as np
from PIL import Image

class BatchAestheticScorer:
    def __init__(self, model_preset="clip_vit_base_patch32"):
        # Load the CLIP components
        self.tokenizer = keras_hub.models.CLIPTokenizer.from_preset(model_preset)
        self.backbone = keras_hub.models.CLIPBackbone.from_preset(model_preset)
        self.preprocessor = keras_hub.layers.CLIPImageConverter.from_preset(model_preset)
        
        # Prepare our "Aesthetic Benchmarks"
        # We compare images against these two concepts
        self.prompt_tokens = self.tokenizer([
            "a professional, award-winning, beautiful photograph", 
            "a blurry, bad, boring, low quality photo"
        ])

    def preprocess_images(self, image_paths):
        """Converts paths into a batch of tensors."""
        processed_images = []
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            # Resize and normalize according to CLIP's requirements
            img_tensor = self.preprocessor(np.array(img))
            processed_images.append(img_tensor)
        return np.array(processed_images)

    def rank_batch(self, image_paths):
        """Processes a list of paths and returns aesthetic scores."""
        images = self.preprocess_images(image_paths)
        
        # Forward pass through CLIP
        outputs = self.backbone({
            "token_ids": self.prompt_tokens,
            "images": images
        })
        
        # logits_per_image is the similarity between each image and the 2 prompts
        logits = outputs["logits_per_image"]
        
        # Apply Softmax to get probabilities
        # Column 0: Prob(Beautiful), Column 1: Prob(Bad)
        probs = keras.activations.softmax(logits, axis=-1).numpy()
        
        # We return the 'Beautiful' probability as the score
        return probs[:, 0]

# Usage Example:
# scorer = BatchAestheticScorer()
# batch = ["img1.jpg", "img2.jpg", "img3.jpg"]
# scores = scorer.rank_batch(batch)
# for path, score in zip(batch, scores):
#     print(f"Path: {path} | Score: {score:.4f}")

e:\DL\highlight_extractor\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## main pipeline

In [9]:
# def main_pipeline(folder_path):
#     # 1. Initialize our tools
#     preprocessor = TripPreprocessor()
#     scorer = BatchAestheticScorer()
    
#     # 2. Run Phase 1 (The Clean-up)
#     # This returns a list of dictionaries with 'path' and 'timestamp'
#     clean_photos = preprocessor.run(folder_path)
    
#     if not clean_photos:
#         print("No high-quality photos found!")
#         return
    
#     # 3. Prepare for Phase 2 (The Scoring)
#     # Extract paths from our clean list to feed to the Scorer
#     paths_to_score = [item['path'] for item in clean_photos]
    
#     # 4. Run Phase 2 in Batches
#     # We process in batches of 32 to keep the GPU/CPU happy
#     all_scores = []
#     batch_size = 32
#     for i in range(0, len(paths_to_score), batch_size):
#         batch = paths_to_score[i : i + batch_size]
#         batch_scores = scorer.rank_batch(batch)
#         all_scores.extend(batch_scores)
    
#     # 5. Combine Results
#     # Add the scores back into our data structure
#     for i, score in enumerate(all_scores):
#         clean_photos[i]['aesthetic_score'] = score

    
#     # ... after scoring ...
#     clusterer = MemoryClusterer(eps_minutes=45) # 45-min gap = new event
#     df_clustered = clusterer.group_by_time(clean_photos)
#     final_highlights = clusterer.fork_highlights(df_clustered, top_n=1)

#     print(f"Original: {len(clean_photos)} | Forked Highlights: {len(final_highlights)}")
        
#     return clean_photos

## pipeline with caching(current)

In [10]:
import json
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class OptimizedTripPipeline:
    def __init__(self, cache_file="trip_cache.json"):
        self.cache_file = cache_file
        self.cache = self.load_cache()

    def load_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, 'r') as f:
                return json.load(f)
        return {}

    def save_cache(self):
        with open(self.cache_file, 'w') as f:
            json.dump(self.cache, f)

    def filter_redundant(self, cluster_df, similarity_threshold=0.92):
        """
        Uses embeddings to ensure we don't pick two photos that look the same.
        """
        # Sort by aesthetic score first
        cluster_df = cluster_df.sort_values(by='aesthetic_score', ascending=False)
        keep_indices = []
        
        # We use the embeddings stored in our data
        # Note: You'll need to store the raw vector in your df to use this
        embeddings = np.array(list(cluster_df['embedding']))
        
        for i in range(len(cluster_df)):
            is_redundant = False
            for j in keep_indices:
                # Calculate Cosine Similarity between current and already kept
                sim = cosine_similarity(embeddings[i].reshape(1, -1), 
                                        embeddings[j].reshape(1, -1))[0][0]
                if sim > similarity_threshold:
                    is_redundant = True
                    break
            
            if not is_redundant:
                keep_indices.append(i)
                
        return cluster_df.iloc[keep_indices]

    def process_with_cache(self, photo_paths, scorer):
        """
        Only runs the CLIP model on photos NOT in the cache.
        """
        results = []
        new_paths = [p for p in photo_paths if p not in self.cache]
        
        # 1. Process New Photos (Phase 2)
        if new_paths:
            print(f"Processing {len(new_paths)} new photos...")
            # Imagine rank_batch also returns embeddings now
            new_scores, new_embeddings = scorer.rank_batch_with_embeddings(new_paths)
            
            for path, score, emb in zip(new_paths, new_scores, new_embeddings):
                self.cache[path] = {
                    "aesthetic_score": float(score),
                    "embedding": emb.tolist() # JSON needs lists, not numpy arrays
                }
        
        # 2. Build Final Dataset from Cache
        for path in photo_paths:
            data = self.cache[path]
            results.append({
                "path": path,
                "aesthetic_score": data["aesthetic_score"],
                "embedding": np.array(data["embedding"])
            })
            
        self.save_cache()
        return results

## memory clustering

In [11]:
import numpy as np
import sklearn
from sklearn.cluster import DBSCAN
import pandas as pd

class MemoryClusterer:
    def __init__(self, eps_minutes=60, min_samples=1):
        # eps is the maximum distance between two samples for them to be in the same cluster
        # We convert minutes to seconds
        self.eps = eps_minutes * 60 
        self.min_samples = min_samples

    def group_by_time(self, photo_data):
        """
        Input: List of dicts from your pipeline [{'path':..., 'timestamp':..., 'aesthetic_score':...}]
        """
        df = pd.DataFrame(photo_data)
        
        # Convert string timestamps to datetime objects, then to unix seconds
        df['dt'] = pd.to_datetime(df['timestamp'], format='%Y:%m:%d %H:%M:%S')
        df['unix'] = df['dt'].view('int64') // 10**9
        
        # We only cluster based on the 'unix' time column
        X = df[['unix']].values
        
        # Fit DBSCAN
        clustering = DBSCAN(eps=self.eps, min_samples=self.min_samples).fit(X)
        df['cluster'] = clustering.labels_
        
        return df

    def fork_highlights(self, df, top_n=1):
        """
        Picks the highest scoring photo(s) from each cluster.
        """
        highlights = []
        for cluster_id in df['cluster'].unique():
            cluster_group = df[df['cluster'] == cluster_id]
            # Sort by aesthetic_score and pick the top N
            top_photos = cluster_group.nlargest(top_n, 'aesthetic_score')
            highlights.extend(top_photos.to_dict('records'))
            
        return highlights